# 04 流式输出

通过 `stream=True` 获取逐块返回的实时输出。

In [1]:
import sys
sys.path.insert(0, '..')

from src.client import get_client
from src.streaming import print_stream

client = get_client()

## 基础流式调用

In [2]:
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "user", "content": "请用 200 字左右介绍深度学习。"}
    ],
    stream=True,
)

print_stream(response)

深度学习是机器学习的一个前沿分支，其核心是通过构建多层人工神经网络来模拟人脑处理信息的方式。它利用“深层”架构，自动从海量数据中逐层提取特征：低层识别边缘和纹理，高层组合出复杂的抽象概念（如人脸或语音）。通过反向传播算法和大量计算资源（如GPU），模型能自我优化参数，从而实现高精度预测。在图像识别、自然语言处理及自动驾驶等领域，深度学习已推动突破性进展，成为人工智能技术的重要引擎。


## 流式 + 累积内容

In [3]:
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "user", "content": "用一句话解释什么是 API。"}
    ],
    stream=True,
)

full_text = ""
for chunk in response:
    delta = chunk.choices[0].delta
    if delta.content:
        full_text += delta.content
        print(delta.content, end="", flush=True)

print()
print(f"\n共 {len(full_text)} 个字符")

API（应用程序编程接口）是定义不同软件组件之间如何相互通信和交换数据的约定集合。

共 41 个字符


## 对流式响应做多轮对话

In [4]:
messages = [
    {"role": "system", "content": "你是一个简洁的助手。"},
    {"role": "user", "content": "什么是图灵测试？"}
]

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
    stream=True,
)

assistant_text = ""
print("Assistant: ", end="", flush=True)
for chunk in response:
    delta = chunk.choices[0].delta
    if delta.content:
        assistant_text += delta.content
        print(delta.content, end="", flush=True)

messages.append({"role": "assistant", "content": assistant_text})
messages.append({"role": "user", "content": "能用一句话总结吗？"})

print()
print()
print("--- 第二轮 ---")
print_stream(client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
    stream=True,
))

Assistant: 图灵测试是由艾伦·图灵于1950年提出的一种测试机器智能的方法。测试中，一名人类评判员通过文本与一台机器和一名人类进行对话，如果评判员无法可靠地区分哪个是机器，则说明机器通过了测试，具备了类似人类的智能。图灵测试的主要目的是检验机器是否能够表现出与人无法区分的智能行为。

--- 第二轮 ---
图灵测试是一种判断机器能否模拟人类智能的测试，如果人类无法通过对话区分机器与人，则机器被认为有智能。
